In [ ]:
from pathlib import Path
import pandas as pd

# Root proyek
PROJECT_DIR = Path.cwd().parent

# Lokasi Dataset 2
DATASET2_DIR = (
    PROJECT_DIR
    / "datasets"
    / "dataset-perkebunan-kalteng"
    / "yolov8"
)

# Lokasi setiap split
SPLITS_2 = {
    "train": DATASET2_DIR / "train",
    "valid": DATASET2_DIR / "valid",
    "test": DATASET2_DIR / "test"
}

IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".webp"
}

results_2 = []

for split_name, split_dir in SPLITS_2.items():

    image_dir = split_dir / "images"
    label_dir = split_dir / "labels"

    if not image_dir.exists() or not label_dir.exists():
        print(f"Struktur folder tidak lengkap: {split_name}")
        continue

    images = [
        file for file in image_dir.rglob("*")
        if file.is_file()
        and file.suffix.lower() in IMAGE_EXTENSIONS
    ]

    labels = [
        file for file in label_dir.rglob("*.txt")
        if file.is_file()
    ]

    total_objects = 0
    empty_labels = 0

    for label in labels:

        lines = [
            line.strip()
            for line in label.read_text(
                encoding="utf-8"
            ).splitlines()
            if line.strip()
        ]

        total_objects += len(lines)

        if not lines:
            empty_labels += 1

    results_2.append({
        "Split": split_name,
        "Jumlah Gambar": len(images),
        "Jumlah File Anotasi": len(labels),
        "Total Objek": total_objects,
        "Anotasi Kosong": empty_labels
    })

eda_summary_2 = pd.DataFrame(results_2)

display(eda_summary_2)

print("\nLokasi Dataset 2:", DATASET2_DIR.resolve())

In [ ]:
from collections import Counter

class_names_2 = [
    "Abnormal",
    "Flower",
    "Ripe",
    "Underripe",
    "Unripe"
]

class_counts_2 = Counter()

split_counts_2 = {}

for split_name, split_dir in SPLITS_2.items():

    counts = Counter()

    label_dir = split_dir / "labels"

    for annotation in label_dir.rglob("*.txt"):

        for line in annotation.read_text(
            encoding="utf-8"
        ).splitlines():

            if not line.strip():
                continue

            values = line.split()
            class_id = int(values[0])

            counts[class_id] += 1
            class_counts_2[class_id] += 1

    split_counts_2[split_name] = counts

class_distribution_2 = pd.DataFrame([
    {
        "Class ID": class_id,
        "Nama Kelas": class_names_2[class_id],
        "Jumlah Objek": class_counts_2[class_id]
    }
    for class_id in range(5)
])

class_distribution_2["Persentase (%)"] = (
    class_distribution_2["Jumlah Objek"]
    / class_distribution_2["Jumlah Objek"].sum()
    * 100
).round(2)

display(class_distribution_2)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(9, 5))

sns.barplot(
    data=class_distribution_2,
    x="Nama Kelas",
    y="Jumlah Objek"
)

plt.title("Distribusi Kelas Dataset Perkebunan Kalimantan Tengah")
plt.xlabel("Kelas TBS")
plt.ylabel("Jumlah Objek")
plt.tight_layout()
plt.show()

In [ ]:
# Distribusi kelas berdasarkan train, valid, dan test

class_names_2 = [
    "Abnormal",
    "Flower",
    "Ripe",
    "Underripe",
    "Unripe"
]

split_distribution_2 = []

for split_name, split_dir in SPLITS_2.items():
    counts = {i: 0 for i in range(len(class_names_2))}

    label_dir = split_dir / "labels"

    for annotation in label_dir.rglob("*.txt"):
        for line in annotation.read_text(encoding="utf-8").splitlines():
            if not line.strip():
                continue

            class_id = int(line.split()[0])
            counts[class_id] += 1

    for class_id, class_name in enumerate(class_names_2):
        split_distribution_2.append({
            "Split": split_name,
            "Nama Kelas": class_name,
            "Jumlah Objek": counts[class_id]
        })

df_split_distribution_2 = pd.DataFrame(split_distribution_2)

display(
    df_split_distribution_2.pivot(
        index="Nama Kelas",
        columns="Split",
        values="Jumlah Objek"
    )
)

In [ ]:
# Validasi format dan nilai bounding box Dataset 2

validation_results_2 = []

for split_name, split_dir in SPLITS_2.items():
    label_dir = split_dir / "labels"

    total_boxes = 0
    valid_boxes = 0
    invalid_format = 0
    invalid_class = 0
    invalid_coordinate = 0
    invalid_size = 0

    for annotation in label_dir.rglob("*.txt"):
        for line in annotation.read_text(encoding="utf-8").splitlines():
            if not line.strip():
                continue

            total_boxes += 1
            values = line.split()

            if len(values) != 5:
                invalid_format += 1
                continue

            try:
                class_id = int(values[0])
                x_center, y_center, width, height = map(float, values[1:])
            except ValueError:
                invalid_format += 1
                continue

            is_valid = True

            if class_id not in range(5):
                invalid_class += 1
                is_valid = False

            if not all(0 <= v <= 1 for v in [
                x_center, y_center, width, height
            ]):
                invalid_coordinate += 1
                is_valid = False

            if width <= 0 or height <= 0:
                invalid_size += 1
                is_valid = False

            if is_valid:
                valid_boxes += 1

    validation_results_2.append({
        "Split": split_name,
        "Total Bounding Box": total_boxes,
        "Valid Bounding Box": valid_boxes,
        "Invalid Format": invalid_format,
        "Invalid Class": invalid_class,
        "Invalid Coordinate": invalid_coordinate,
        "Invalid Size": invalid_size
    })

df_validation_2 = pd.DataFrame(validation_results_2)
display(df_validation_2)

In [ ]:
# Validasi batas bounding box Dataset 2

boundary_results_2 = []

for split_name, split_dir in SPLITS_2.items():
    label_dir = split_dir / "labels"

    total_boxes = 0
    boxes_outside = 0

    for annotation in label_dir.rglob("*.txt"):
        for line in annotation.read_text(encoding="utf-8").splitlines():
            if not line.strip():
                continue

            values = line.split()

            if len(values) != 5:
                continue

            class_id = int(values[0])
            x_center, y_center, width, height = map(float, values[1:])

            total_boxes += 1

            x_min = x_center - width / 2
            y_min = y_center - height / 2
            x_max = x_center + width / 2
            y_max = y_center + height / 2

            if (
                x_min < 0 or y_min < 0
                or x_max > 1 or y_max > 1
            ):
                boxes_outside += 1

    boundary_results_2.append({
        "Split": split_name,
        "Total Bounding Box": total_boxes,
        "Bounding Box di Luar Batas": boxes_outside,
        "Bounding Box dalam Batas": total_boxes - boxes_outside
    })

df_boundary_2 = pd.DataFrame(boundary_results_2)
display(df_boundary_2)

In [ ]:
import cv2
import matplotlib.pyplot as plt
import random

# Nama kelas
class_names_2 = [
    "Abnormal",
    "Flower",
    "Ripe",
    "Underripe",
    "Unripe"
]

# Ambil beberapa gambar dari train
image_dir = SPLITS_2["train"] / "images"
label_dir = SPLITS_2["train"] / "labels"

image_files = list(image_dir.glob("*"))

# Pilih 6 gambar secara acak
random.seed(42)
selected_images = random.sample(image_files, min(6, len(image_files)))

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for ax, image_path in zip(axes, selected_images):

    image = cv2.imread(str(image_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    height, width = image.shape[:2]

    label_path = label_dir / f"{image_path.stem}.txt"

    if label_path.exists():
        lines = label_path.read_text(encoding="utf-8").splitlines()

        for line in lines:
            if not line.strip():
                continue

            class_id, x_center, y_center, box_width, box_height = (
                map(float, line.split())
            )

            x_center *= width
            y_center *= height
            box_width *= width
            box_height *= height

            x1 = int(x_center - box_width / 2)
            y1 = int(y_center - box_height / 2)
            x2 = int(x_center + box_width / 2)
            y2 = int(y_center + box_height / 2)

            cv2.rectangle(
                image,
                (x1, y1),
                (x2, y2),
                (255, 0, 0),
                2
            )

            label = class_names_2[int(class_id)]

            cv2.putText(
                image,
                label,
                (x1, max(y1 - 5, 15)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                (255, 0, 0),
                2
            )

    ax.imshow(image)
    ax.set_title(image_path.name)
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Pemeriksaan dimensi gambar Dataset 2

from PIL import Image
from collections import Counter
from tqdm import tqdm

dimension_results_2 = []

for split_name, split_dir in SPLITS_2.items():
    image_dir = split_dir / "images"

    for image_path in tqdm(
        list(image_dir.glob("*")),
        desc=f"Memeriksa {split_name}"
    ):
        try:
            with Image.open(image_path) as img:
                width, height = img.size

                dimension_results_2.append({
                    "Split": split_name,
                    "Nama File": image_path.name,
                    "Width": width,
                    "Height": height,
                    "Format": img.format
                })

        except Exception as e:
            print(f"Gagal membaca {image_path.name}: {e}")

df_dimensions_2 = pd.DataFrame(dimension_results_2)

print("Total gambar berhasil diperiksa:", len(df_dimensions_2))
print("\nDistribusi dimensi gambar:")

display(
    df_dimensions_2.groupby(
        ["Width", "Height"]
    ).size().reset_index(name="Jumlah Gambar")
)

In [ ]:
# Pemeriksaan kualitas gambar Dataset 2

import cv2
import numpy as np
from tqdm import tqdm

quality_results_2 = []

for split_name, split_dir in SPLITS_2.items():
    image_dir = split_dir / "images"

    image_files = list(image_dir.glob("*"))

    for image_path in tqdm(
        image_files,
        desc=f"Mengukur kualitas {split_name}"
    ):
        image = cv2.imread(str(image_path))

        if image is None:
            continue

        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

        brightness = np.mean(gray)
        contrast = np.std(gray)
        sharpness = cv2.Laplacian(
            gray,
            cv2.CV_64F
        ).var()

        quality_results_2.append({
            "Split": split_name,
            "Nama File": image_path.name,
            "Brightness": brightness,
            "Contrast": contrast,
            "Sharpness": sharpness
        })

df_quality_2 = pd.DataFrame(quality_results_2)

print("Total gambar berhasil dianalisis:", len(df_quality_2))

display(
    df_quality_2.groupby("Split")[
        ["Brightness", "Contrast", "Sharpness"]
    ].mean().round(2)
)

In [ ]:
# Visualisasi distribusi kualitas gambar Dataset 2

import seaborn as sns
import matplotlib.pyplot as plt

metrics_2 = [
    "Brightness",
    "Contrast",
    "Sharpness"
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, metric in zip(axes, metrics_2):
    sns.boxplot(
        data=df_quality_2,
        x="Split",
        y=metric,
        ax=ax
    )

    ax.set_title(f"Distribusi {metric}")
    ax.set_xlabel("Split")
    ax.set_ylabel(metric)

plt.suptitle(
    "Distribusi Kualitas Gambar Dataset Perkebunan Kalteng",
    fontsize=14
)

plt.tight_layout()
plt.show()

In [ ]:
# Simpan hasil EDA Dataset 2

from pathlib import Path

RESULT_DIR_2 = PROJECT_DIR / "results" / "eda_dataset_perkebunan"
RESULT_DIR_2.mkdir(parents=True, exist_ok=True)

# 1. Distribusi kelas keseluruhan
class_distribution_2.to_csv(
    RESULT_DIR_2 / "distribusi_kelas.csv",
    index=False
)

# 2. Distribusi kelas per split
df_split_distribution_2.to_csv(
    RESULT_DIR_2 / "distribusi_kelas_per_split.csv",
    index=False
)

# 3. Validasi bounding box
df_validation_2.to_csv(
    RESULT_DIR_2 / "validasi_bounding_box.csv",
    index=False
)

# 4. Validasi batas bounding box
df_boundary_2.to_csv(
    RESULT_DIR_2 / "validasi_batas_bounding_box.csv",
    index=False
)

# 5. Dimensi gambar
df_dimensions_2.to_csv(
    RESULT_DIR_2 / "dimensi_gambar.csv",
    index=False
)

# 6. Kualitas gambar
df_quality_2.to_csv(
    RESULT_DIR_2 / "kualitas_gambar.csv",
    index=False
)

print("Seluruh hasil EDA Dataset 2 berhasil disimpan.")
print("Lokasi:", RESULT_DIR_2)